In [ ]:
load_ext jupyter_black

In [ ]:
import matplotlib.pyplot as plt
import os
import pickle
import pandas as pd
import seaborn as sns
from deepsig import aso

In [ ]:
demographics = {
    "cad": [
        "annotator_age",
        "annotator_education_level",
        "annotator_gender",
        "annotator_political",
    ],
    "cad_en": [
        "annotator_age",
        "annotator_education_level",
        "annotator_gender",
        "annotator_political",
        "annotator_ethnicity",
    ],
    "cad_hi": [
        "annotator_age",
        "annotator_education_level",
        "annotator_gender",
        "annotator_political",
        "annotator_ethnicity",
    ],
    "cad_pt": [
        "annotator_age",
        "annotator_education_level",
        "annotator_gender",
        "annotator_political",
        "annotator_ethnicity",
    ],
    "cad_fr": [
        "annotator_age",
        "annotator_education_level",
        "annotator_gender",
        "annotator_political",
        "annotator_ethnicity",
    ],
    "cad_it": [
        "annotator_age",
        "annotator_education_level",
        "annotator_gender",
        "annotator_political",
        "annotator_ethnicity",
    ],
    "chen": ["label"],
    "prism": [
        "age",
        "birth_region",
        "education",
        "employment_status",
        "english_proficiency",
        "ethnicity",
        "gender",
        "lm_familiarity",
        "marital_status",
        "religion",
        "reside_region",
    ],
}

In [ ]:
belief_cols = {
    "human_": ["Gender"],
    "revealed_": ["Gender"],
    "shared_extracted_": [
        "Age",
        "Educational Background",
        "Gender",
        "English proficiency",
        "Ethnicity",
        "Socioeconomic Status",
        "Marital Status",
        "Religion",
    ],
    "value_JSON_": [
        "Age",
        "Educational Background",
        "Gender",
        "English proficiency",
        "Ethnicity",
        "Socioeconomic Status",
        "Marital Status",
        "Religion",
    ],
    "unknown_token_": [
        "Age",
        "Educational Background",
        "Gender",
        "English proficiency",
        "Ethnicity",
        "Socioeconomic Status",
        "Marital Status",
        "Religion",
    ],
}

In [ ]:
erasure = {
    "prism": {
        "age": ["e_remorse_user_prompt", "e_anger_user_prompt"],
        "gender": ["e_remorse_user_prompt", "type_to_token_ratio_user_prompt"],
        "ethnicity": ["topic:diet,healthy snacks,plant-based", "e_grief_user_prompt"],
    },
    "cad_en": {
        "annotator_age": [
            # "e_disappointment_model_response",
            # "model_response_liwc_filler",
            # "e_disappointment_model_response_model_response_liwc_filler",
            "e_embarrassment_user_prompt",
            "avg_sent_len_model_response",
            "e_embarrassment_user_prompt_avg_sent_len_model_response",
        ],
        "annotator_gender": [
            # "e_realization_model_response",
            # "user_prompt_liwc_filler",
            # "e_realization_model_response_user_prompt_liwc_filler",
            "e_embarrassment_user_prompt",
            "avg_sent_len_model_response",
            "e_embarrassment_user_prompt_avg_sent_len_model_response",
        ],
    },
}

In [ ]:
data = {
    "dataset": [],
    "demographic": [],
    "balanced": [],
    "layer": [],
    "type": [],
    "cv": [],
    "f1": [],
    "majority f1": [],
    "random f1": [],
    "majority aso": [],
    "random aso": [],
    "majority aso_better": [],
    "random aso_better": [],
    "belief": [],
    "item": [],
}

for dataset in erasure:
    for demographic in erasure[dataset]:
        for item in erasure[dataset][demographic]:
            for balanced in [True, False]:
                with open(
                    f"llama_erasure/Llama-3.1-8B-Instruct_{dataset}_dim_{demographic}{'balanced' if balanced else ''}_{item}_mlp_scores.pkl",
                    "rb",
                ) as infile:
                    results = pickle.load(infile)
                for layer in results:
                    for cv in range(5):
                        data["dataset"].append(dataset)
                        data["demographic"].append(demographic)
                        data["balanced"].append(balanced)
                        data["layer"].append(layer)
                        data["cv"].append(cv)
                        data["type"].append("mlp")
                        for score in ["f1", "majority f1", "random f1"]:
                            data[score].append(results[layer][score][cv])
                        for aso_score in ["majority aso", "random aso"]:
                            data[aso_score].append(results[layer][aso_score])
                            data[aso_score + "_better"].append(
                                results[layer][aso_score] < 0.5
                            )
                        data["belief"].append(False)
                        data["item"].append(item)

for dataset in demographics:
    for demographic in demographics[dataset]:
        for balanced in [True, False]:
            if os.path.isfile(
                f"llama_probing_results/Llama-3.1-8B-Instruct_{dataset}_{demographic}{'_balanced' if balanced else ''}_mlp_scores.pkl"
            ):
                with open(
                    f"llama_probing_results/Llama-3.1-8B-Instruct_{dataset}_{demographic}{'_balanced' if balanced else ''}_scores.pkl",
                    "rb",
                ) as infile:
                    results = pickle.load(infile)
                for layer in results:
                    for cv in range(5):
                        data["dataset"].append(dataset)
                        data["demographic"].append(demographic)
                        data["balanced"].append(balanced)
                        data["layer"].append(layer)
                        data["cv"].append(cv)
                        data["type"].append("mlp")
                        for score in ["f1", "majority f1", "random f1"]:
                            data[score].append(results[layer][score][cv])
                        for aso_score in ["majority aso", "random aso"]:
                            data[aso_score].append(results[layer][aso_score])
                            data[aso_score + "_better"].append(
                                results[layer][aso_score] < 0.5
                            )
                        data["belief"].append(False)
                        data["item"].append(None)

            with open(
                f"llama_probing_results/Llama-3.1-8B-Instruct_{dataset}_{demographic}{'_balanced' if balanced else ''}_scores.pkl",
                "rb",
            ) as infile:
                results = pickle.load(infile)
            for layer in results:
                for cv in range(5):
                    data["dataset"].append(dataset)
                    data["demographic"].append(demographic)
                    data["balanced"].append(balanced)
                    data["layer"].append(layer)
                    data["cv"].append(cv)
                    data["type"].append("linear")
                    for score in ["f1", "majority f1", "random f1"]:
                        data[score].append(results[layer][score][cv])
                    for aso_score in ["majority aso", "random aso"]:
                        data[aso_score].append(results[layer][aso_score])
                        data[aso_score + "_better"].append(
                            results[layer][aso_score] < 0.5
                        )
                    data["belief"].append(False)
                    data["item"].append(None)

            with open(
                f"llama_probing_results/Llama-3.1-8B-Instruct_{dataset}_{demographic}{'_balanced' if balanced else ''}_scores.pkl",
                "rb",
            ) as infile:
                results = pickle.load(infile)
            for layer in results:
                for cv in range(5):
                    data["dataset"].append(dataset)
                    data["demographic"].append(demographic)
                    data["balanced"].append(balanced)
                    data["layer"].append(layer)
                    data["cv"].append(cv)
                    data["type"].append("linear")
                    for score in ["f1", "majority f1", "random f1"]:
                        data[score].append(results[layer][score][cv])
                    for aso_score in ["majority aso", "random aso"]:
                        data[aso_score].append(results[layer][aso_score])
                        data[aso_score + "_better"].append(
                            results[layer][aso_score] < 0.5
                        )
                    data["belief"].append(False)
                    data["item"].append(None)

    if dataset in ["chen", "prism", "cad_en"]:
        for belief_col in belief_cols:
            if belief_col == "human_" and dataset != "chen":
                continue
            for demographic in belief_cols[belief_col]:
                for balanced in [True, False]:
                    with open(
                        f"llama_probing_results/Llama-3.1-8B-Instruct_{dataset}_{belief_col}{demographic.replace(' ','')}{'_balanced' if balanced else ''}_scores.pkl",
                        "rb",
                    ) as infile:
                        results = pickle.load(infile)
                    for layer in results:
                        for cv in range(5):
                            data["dataset"].append(dataset)
                            data["demographic"].append(demographic)
                            data["balanced"].append(balanced)
                            data["layer"].append(layer)
                            data["cv"].append(cv)
                            data["type"].append("linear")
                            for score in ["f1", "majority f1", "random f1"]:
                                data[score].append(results[layer][score][cv])
                            for aso_score in ["majority aso", "random aso"]:
                                data[aso_score].append(results[layer][aso_score])
                                data[aso_score + "_better"].append(
                                    results[layer][aso_score] < 0.5
                                )
                            data["belief"].append(belief_col)
                            data["item"].append(None)

df = pd.DataFrame(data)

In [ ]:
df

In [ ]:
def line_plot(df, ax, legend, df2s=[]):
    sns.lineplot(
        df,
        x="layer",
        y="f1",
        ax=ax,
        legend=legend,
        label="Probe",
    )
    for df2 in df2s:
        item = df2["item"].unique()[0]
        sns.lineplot(
            df2,
            x="layer",
            y="f1",
            ax=ax,
            legend=legend,
            label=f"Probe - {item}",
        )
    sns.lineplot(
        df,
        x="layer",
        y="majority f1",
        ax=ax,
        legend=legend,
        label="Majority",
    )
    sns.lineplot(
        df,
        x="layer",
        y="random f1",
        ax=ax,
        legend=legend,
        label="Random",
    )
    if legend:
        if len(df2s) > 0:
            ax.legend(loc="best")
        else:
            ax.legend(loc="center left", bbox_to_anchor=(1, 0.5))

    if len(df2s) > 0:
        for df2 in df2s:
            for layer in df2["layer"].unique():
                better = (
                    aso(
                        df.loc[df["layer"] == layer, "f1"],
                        df2.loc[df2["layer"] == layer, "f1"],
                        seed=42,
                    )
                    < 0.5
                )
                ax.scatter(
                    layer,
                    df2.loc[(df2["layer"] == layer)]["f1"].mean(),
                    marker="o" if better else "x",
                    color="blue" if better else "red",
                )

    else:
        better_than_baseline = (
            df.groupby("layer")[["majority aso_better", "random aso_better"]]
            .min()
            .min(axis=1)
        )

        m = ["o" if x else "x" for x in better_than_baseline]
        c = ["blue" if x else "red" for x in better_than_baseline]

        for layer in df["layer"].unique():
            ax.scatter(
                layer,
                df.loc[(df["layer"] == layer)]["f1"].mean(),
                marker=m[layer],
                color=c[layer],
            )
    ax.set_ylabel("Macro F1")

In [ ]:
for dataset in erasure:
    # if not os.path.isfile(f"probing_figures/{dataset}_dim_notbalanced.png"):
    #     fig, axes = plt.subplots(
    #         1,
    #         len(erasure[dataset]),
    #         figsize=(5 * len(erasure[dataset]), 8),
    #         sharey=True,
    #     )
    #     for d, demographic in enumerate(erasure[dataset]):
    #         if len(erasure[dataset]) > 1:
    #             ax = axes[d]
    #         else:
    #             ax = axes
    #         line_plot(
    #             df.loc[
    #                 (df["dataset"] == dataset)
    #                 & (df["demographic"] == demographic)
    #                 & (df["type"] == "mlp")
    #                 & (df["balanced"] == False)
    #                 & (df["belief"] == False)
    #                 & pd.isna(df["item"])
    #             ],
    #             ax,
    #             legend=True,
    #             df2s=[
    #                 df.loc[
    #                     (df["dataset"] == dataset)
    #                     & (df["demographic"] == demographic)
    #                     & (df["type"] == "mlp")
    #                     & (df["balanced"] == False)
    #                     & (df["belief"] == False)
    #                     & (df["item"] == item)
    #                 ]
    #                 for item in erasure[dataset][demographic]
    #             ],
    #         )
    #         ax.set_title(demographic)
    #     fig.suptitle(dataset + " - Not Balanced")
    #     plt.savefig(
    #         f"probing_figures/{dataset}_dim_notbalanced.png", bbox_inches="tight"
    #     )
    #     plt.show()

    if not os.path.isfile(f"probing_figures/{dataset}_dim_balanced.png"):
        fig2, axes = plt.subplots(
            1,
            len(erasure[dataset]),
            figsize=(5 * len(erasure[dataset]), 8),
            sharey=True,
        )
        for d, demographic in enumerate(erasure[dataset]):
            if len(erasure[dataset]) > 1:
                ax = axes[d]
            else:
                ax = axes
            line_plot(
                df.loc[
                    (df["dataset"] == dataset)
                    & (df["demographic"] == demographic)
                    & (df["type"] == "linear")
                    & (df["balanced"] == True)
                    & (df["belief"] == False)
                    & pd.isna(df["item"])
                ],
                ax,
                legend=True,
                df2s=[
                    df.loc[
                        (df["dataset"] == dataset)
                        & (df["demographic"] == demographic)
                        & (df["type"] == "mlp")
                        & (df["balanced"] == True)
                        & (df["belief"] == False)
                        & (df["item"] == item)
                    ]
                    for item in erasure[dataset][demographic]
                ],
            )
            ax.set_title(demographic)
        fig2.suptitle(dataset + " - Balanced")
        plt.savefig(f"probing_figures/{dataset}_dim_balanced.png", bbox_inches="tight")
        plt.show()

In [ ]:
for dataset in demographics:
    fig, axes = plt.subplots(
        1,
        len(demographics[dataset]),
        figsize=(5 * len(demographics[dataset]), 4),
        sharey=True,
    )
    for d, demographic in enumerate(demographics[dataset]):
        if len(demographics[dataset]) > 1:
            ax = axes[d]
        else:
            ax = axes
        line_plot(
            df.loc[
                (df["dataset"] == dataset)
                & (df["demographic"] == demographic)
                & (df["type"] == "linear")
                & (df["balanced"] == False)
                & (df["belief"] == False)
            ],
            ax,
            legend=d == len(demographics[dataset]) - 1,
        )
        ax.set_title(demographic)
    fig.suptitle(dataset + " - Not Balanced")
    plt.savefig(f"probing_figures/{dataset}_notbalanced.png")
    plt.show()
    fig2, axes = plt.subplots(
        1,
        len(demographics[dataset]),
        figsize=(5 * len(demographics[dataset]), 4),
        sharey=True,
    )
    for d, demographic in enumerate(demographics[dataset]):
        if len(demographics[dataset]) > 1:
            ax = axes[d]
        else:
            ax = axes
        line_plot(
            df.loc[
                (df["dataset"] == dataset)
                & (df["demographic"] == demographic)
                & (df["type"] == "linear")
                & (df["balanced"] == True)
                & (df["belief"] == False)
            ],
            ax,
            legend=d == len(demographics[dataset]) - 1,
        )
        ax.set_title(demographic)
    fig2.suptitle(dataset + " - Balanced")
    plt.savefig(f"probing_figures/{dataset}_balanced.png")
    plt.show()

In [ ]:
for dataset in demographics:
    if dataset in ["chen", "prism", "cad_en"]:
        for belief_col in belief_cols:
            if belief_col == "human_" and dataset != "chen":
                continue
            fig, axes = plt.subplots(
                1,
                len(belief_cols[belief_col]),
                figsize=(5 * len(belief_cols[belief_col]), 4),
                sharey=True,
            )
            for d, demographic in enumerate(belief_cols[belief_col]):
                if len(belief_cols[belief_col]) > 1:
                    ax = axes[d]
                else:
                    ax = axes
                line_plot(
                    df.loc[
                        (df["dataset"] == dataset)
                        & (df["demographic"] == demographic)
                        & (df["type"] == "linear")
                        & (df["balanced"] == False)
                        & (df["belief"] == belief_col)
                    ],
                    ax,
                    legend=d == len(belief_cols[belief_col]) - 1,
                )
                ax.set_title(demographic)
            fig.suptitle(dataset + f" - {belief_col}" + " - Not Balanced")
            plt.savefig(f"probing_figures/{dataset}_{belief_col}notbalanced.png")
            plt.show()
            fig2, axes = plt.subplots(
                1,
                len(belief_cols[belief_col]),
                figsize=(5 * len(belief_cols[belief_col]), 4),
                sharey=True,
            )
            for d, demographic in enumerate(belief_cols[belief_col]):
                if len(belief_cols[belief_col]) > 1:
                    ax = axes[d]
                else:
                    ax = axes
                line_plot(
                    df.loc[
                        (df["dataset"] == dataset)
                        & (df["demographic"] == demographic)
                        & (df["type"] == "linear")
                        & (df["balanced"] == True)
                        & (df["belief"] == belief_col)
                    ],
                    ax,
                    legend=d == len(belief_cols[belief_col]) - 1,
                )
                ax.set_title(demographic)
            fig2.suptitle(dataset + f" - {belief_col}" + " - Balanced")
            plt.savefig(f"probing_figures/{dataset}_{belief_col}balanced.png")
            plt.show()